# 07 · Fiber S1 from nexus vs SiPM dark counts

**Purpose.** detsim's fiber S1 is a placeholder (03). But the nexus files already record the S1 scintillation light detected by the 108 fiber sensors (`/MC/sns_response`, sensor `F_SENSOR`). Here: how big is that S1, how is it spread in time, and can it be seen above the dark counts of the SiPMs?

**Inputs.** nexus Kr files (`$NEXUS_DIR/Kr/20250624_Next100_1Kev_ELoff_Kr_<i>.next.h5`): `/MC/sns_response`, `/MC/sns_positions`, `/MC/hits`.

**Outputs.** None (numbers and plots).

**What is checked.** S1 pes per Kr event and its z dependence; arrival-time spread (sets the integration window); expected dark counts in that window from the datasheet DCR (2000 kcps typ. per 6×6 mm² S13360-6075PE at 25 °C, Hamamatsu product page) and from your CIGAR DCR; separation of S1 from dark noise for Kr (41.5 keV) and, scaled, for 0νββ (2458 keV).

**Assumptions to verify.** One 6×6 mm² SiPM per fiber channel (108 channels); nexus applies the SiPM PDE to `F_SENSOR` (`sensor_type = SiPM_Hamamatsu`); S1 scales linearly with energy (for the 0νββ estimate).

**Next step.** If S1 is usable, replace the detsim placeholder with a fiber S1 table built from these files.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import poisson
import warnings; warnings.filterwarnings("ignore")
from icana import paths, checks

In [ ]:
SAMPLE, N_FILES = "20250624_Next100_1Kev_ELoff_Kr", 5          # 5 x 1000 Kr events
files = paths.nexus_files(SAMPLE)[:N_FILES]
s1_ev, t_all, z_ev = [], [], []
for f in files:
    sns  = pd.read_hdf(f, "/MC/sns_positions")
    fib  = sns.loc[sns.sensor_name == "F_SENSOR", "sensor_id"]
    resp = pd.read_hdf(f, "/MC/sns_response"); resp = resp[resp.sensor_id.isin(fib)]
    cfg  = pd.read_hdf(f, "/MC/configuration")
    binw_ns = float(cfg.loc[cfg.param_key == "F_SENSOR_binning", "param_value"].iloc[0].split()[0]) * 1e3   # mus -> ns
    hits = pd.read_hdf(f, "/MC/hits")
    q = resp.groupby("event_id").charge.sum()
    z = hits.groupby("event_id").apply(lambda h: np.average(h.z, weights=h.energy))
    s1_ev.append(q.reindex(z.index, fill_value=0)); z_ev.append(z)
    t_all.append(np.repeat(resp.time_bin.values * binw_ns, resp.charge.values.astype(int)))
s1 = pd.concat(s1_ev, ignore_index=True); z = pd.concat(z_ev, ignore_index=True); t = np.concatenate(t_all)
print(f"{len(s1)} Kr events, {len(fib)} fiber sensors, time bin {binw_ns} ns")
print(f"S1 on the fibers: mean {s1.mean():.1f} pes, median {s1.median():.0f}, rms {s1.std():.1f}; events with 0 pes: {(s1==0).mean()*100:.2f} %")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].hist(s1, np.arange(0, s1.max() + 2) - 0.5); ax[0].set_xlabel("S1 per Kr event (pes, 108 fibers)"); ax[0].set_ylabel("events")
zb = np.linspace(0, 1200, 13); zc = 0.5 * (zb[1:] + zb[:-1]); idx = np.digitize(z, zb) - 1
m = [s1[idx == k].mean() for k in range(len(zc))]
ax[1].plot(zc, m, "o-"); ax[1].set_xlabel("z of the deposit (mm)"); ax[1].set_ylabel("mean S1 (pes)")
ax[2].hist(t, np.linspace(0, 1000, 101)); ax[2].set_xlabel("arrival time after the decay (ns)"); ax[2].set_ylabel("pes")
plt.tight_layout()
q = np.percentile(t, [50, 90, 95, 99])
print("S1 arrival time: 50/90/95/99 % of the light by", ", ".join(f"{v:.0f} ns" for v in q))

## Dark counts in the S1 window
Dark pulses in a window of length *w* summed over all channels: *B* = DCR × N_channels × w, multiplied by the mean avalanches per dark pulse 1/(1−λ_CN) if crosstalk is included (Vinogradov's branching Poisson, as in CIGAR).
Fill `DCR_CIGAR_HZ` and `LAMBDA_CN` with your CIGAR results at the operating temperature.

In [ ]:
N_CH          = 108
DCR_DATASHEET = 2.0e6       # Hz per 6x6 mm2 SiPM, typ., 25 C (Hamamatsu S13360-6075PE product page)
DCR_CIGAR_HZ  = None        # <- your measured primary DCR per SiPM at the operating temperature
LAMBDA_CN     = 0.0         # <- your CIGAR correlated-noise parameter (0 = no crosstalk)
E_KR, E_QBB   = 41.5, 2458.0

def table(s1_mean, dcr, windows_ns=(100, 200, 300, 500, 1000, 2000)):
    rows = []
    for w in windows_ns:
        frac = (t <= w).mean()                      # fraction of S1 light inside the window (window opens at the decay)
        S = s1_mean * frac
        B = dcr * N_CH * w * 1e-9 / (1 - LAMBDA_CN)
        thr = poisson.ppf(1 - 1e-3, B) + 1          # threshold with 0.1 % of pure-noise windows above it
        eff = 1 - poisson.cdf(thr - 1, S + B)       # fraction of S1 windows above that threshold
        rows.append(dict(window_ns=w, S1_in_window=round(S, 1), dark_B=round(B, 1), S_over_sqrtB=round(S / np.sqrt(B), 2) if B else np.inf,
                         threshold_pes=int(thr), S1_efficiency=round(eff, 3)))
    return pd.DataFrame(rows)

for label, dcr in [("datasheet, 25 C", DCR_DATASHEET)] + ([("CIGAR", DCR_CIGAR_HZ)] if DCR_CIGAR_HZ else []):
    print(f"\n=== Kr (41.5 keV), DCR {label}: {dcr/1e3:.0f} kHz per SiPM")
    print(table(s1.mean(), dcr).to_string(index=False))
    print(f"=== 0nubb (2458 keV, S1 scaled linearly: {s1.mean()*E_QBB/E_KR:.0f} pes), DCR {label}")
    print(table(s1.mean() * E_QBB / E_KR, dcr).to_string(index=False))

In [ ]:
# DCR needed so that Kr S1 is found in 90 % of events with a 300 ns window (0.1 % noise windows above threshold)
for dcr in [2e6, 1e6, 5e5, 2e5, 1e5, 5e4]:
    r = table(s1.mean(), dcr, windows_ns=(300,)).iloc[0]
    print(f"DCR {dcr/1e3:6.0f} kHz/SiPM: dark {r.dark_B:6.1f} pes in 300 ns, S1 efficiency {r.S1_efficiency:.2f}")

### Reading the result
- S1 for Kr is a few tens of pes spread over ~0.1–1 µs, while the datasheet DCR gives a dark background of the same size or larger in that window at 25 °C: Kr S1 needs cooled SiPMs (your CIGAR DCR vs temperature gives the real number).
- For 0νββ, S1 is ~60× larger and stands out even at the datasheet DCR, so fibers can give t0 for physics events; the limitation is the Kr calibration.
- These are expectations from the simulation, not measurements. Verify the assumptions listed at the top before quoting them.